### 1. validate_call
it will check the type of the input and validate it 

In [2]:
from pydantic import validate_call

@validate_call
def add_number(a:int, b:int):
    return a+b

res = add_number("10", 2)
print(res)

12


### 2. StrictInt 
to check the number should be in the int

In [3]:
from pydantic import StrictInt

@validate_call
def add_number(a:StrictInt, b:StrictInt):
    return a+b

res = add_number("10", 2)
print(res)

ValidationError: 1 validation error for add_number
0
  Input should be a valid integer [type=int_type, input_value='10', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_type

### 3. Class Validation

In [4]:
from pydantic import BaseModel

class User(BaseModel):
    user_id: int
    name: str
    age: int
    is_active: bool
    
user = User(
    user_id= 1,
    name= "Bruce Banner",
    age= 25,
    is_active= True    
)

print(user)

user_id=1 name='Bruce Banner' age=25 is_active=True


### 4. Types in validation

In [5]:
from pydantic import PositiveInt
from typing import List, Literal, Optional

class Orders(BaseModel):
    user_id: PositiveInt
    product_ids: List[int]
    payment_mode: Literal["Online", "COD", "EMI"]
    delivery_note: Optional[str] = None
    
order = Orders(
    user_id= 1,
    product_ids= [102,101,204],
    payment_mode="Online",
)

print(order)    

user_id=1 product_ids=[102, 101, 204] payment_mode='Online' delivery_note=None


### 5. Field 

In [8]:
from pydantic import Field

class Employee(BaseModel):
    name:  str  = Field(
        ..., # represent it is required
        min_length= 5,
        max_length=20,
        description= "Employee Name"
    )
    salary: int = Field(
        ...,
        ge= 100,
        le= 2000
    )
    email:str = Field(
        default="" # if it is optional not required 
    )
    
    
employee = Employee(
    name="Hello",
    salary= 2000
)

print(employee)

name='Hello' salary=2000 email=''


### 6. Field and model validator

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator

class Register(BaseModel):
    username: str = Field(..., min_length=6)
    password: str
    re_password: str
    mobile: int

    @field_validator("mobile") # with the help of this we can validate a particular field at a time 
    def validate_mobile(cls, value):
        if len(str(value)) != 10:
            return ValueError("Mobile number must be 10 digit")
        return value
    
    @model_validator(mode="after") # with this we can validate more than one field at the same time 
    def pass_validator(self):
        if self.password != self.re_password:
            return ValueError("Password do not match")
        
        if len(self.password) < 8:
            return ValueError("Password min length is 8") 
        return self
        
new_register = Register(
    username="Brucase",
    password="1234112342",
    re_password= "1234112",
    mobile= 1234112342
)

print(new_register)
    


username='Brucase' password='1234112342' re_password='1234112' mobile=1234112342


/var/folders/tr/wd0yb2ws5pg26ymd3yj6pbr00000gn/T/ipykernel_14871/52670172.py:24: UserWarning: A custom validator is returning a value other than `self`.
Returning anything other than `self` from a top level model validator isn't supported when validating via `__init__`.
See the `model_validator` docs (https://pydantic.dev/docs/validation/2.14/concepts/validators/#model-validators) for more details.
  new_register = Register(


### 7. Computed

In [32]:
from pydantic import computed_field

class CartItems(BaseModel):
    user_id: int
    product_id: int
    price: int
    quantity: int
    
    @computed_field
    @property
    def total_amount(self) -> int:
        return self.price * self.quantity
    
    
cart = CartItems(
    user_id=1,
    product_id=1,
    price=120,
    quantity=4
)

print(cart)

user_id=1 product_id=1 price=120 quantity=4 total_amount=480


### 8. Annotated 

it help us to provide the types with extra checks with it 

In [35]:
from typing import Annotated

class User(BaseModel):
    name: Annotated[str, Field(..., min_length=8)]
    age: Annotated[int, Field(..., ge=18)]
    
user = User(name="hello world", age=1)
print(user) 

ValidationError: 1 validation error for User
age
  Input should be greater than or equal to 18 [type=greater_than_equal, input_value=1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal

### 9. Sterilize and De-Sterilize

- Sterilization: converting pydantic class to dictionary form (Pydantic -> dict)

- De-Sterilization: converting dictionary form to pydantic class (dict -> Pydantic)

In [39]:
class Students(BaseModel):
    name: str
    email:str
    age: int
    branch: str
    sem: int
    
    
data = {
    "name": "rahul",
    "email": "rahul@gmail.com",
    "age": 31,
    "branch": "CS",
    "sem": 3
}

# student = Students(**data)
student = Students.model_validate(data) # de-serialization  ie breaking dictionary into pydantic class
print(student)
print(student.model_dump()) # serialization ie creating pydantic class to dictionary form again

name='rahul' email='rahul@gmail.com' age=31 branch='CS' sem=3
{'name': 'rahul', 'email': 'rahul@gmail.com', 'age': 31, 'branch': 'CS', 'sem': 3}
